In [0]:
##############  ENTRANCES  #################

# diligenciar el nombre de la tabla completo en 'table_name'. Una vez diligenciado, ejecutar este comando con ctrl + Enter, luego, pasamos al siguiente comando con Alt + tecla ⬇ (flecha hacia abajo)
table_name = 'prod_latam_catalog.crm_reporting.fact_gdm_beauty_profile'


##### variables derivadas de table
table = spark.table(table_name)
# Definir el id a usar en funcion del contenido de 'table_name'
id_key = "source_customer_id" if 'fact_gdm_beauty_profile' in table_name else "brand_mdm_id"



############  FUNCTIONS  ###############
# Import packages and functions needed 
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.types import StringType


# Function that identifies and explodes array and struct fields
def expand_arrays(df):
    array_columns = []
    
    # search and select the array columns
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # loop the array columns
    for col_name in array_columns:
        # Explode each array column
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verify whether the data type of the array is a struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # If it does, extract the struct subfields, create new fields and delete the original one
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # if the data type is not StructType (ie, StringType o DoubleType), replace the original column with the exploded one
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # delete the temporal exploded field
        df = df.drop(f"exploded_{col_name}")
    
    return df


# Función para separar el campo usando '~' y ',' como delimitadores
def split_multiple_delimiters(df, input_col, output_col):
    # Usamos regexp_replace para normalizar los delimitadores a uno solo (por ejemplo ',')
    normalized_col = F.regexp_replace(input_col, '[~,]+', ',')
    # Hacemos split del resultado normalizado
    return df.withColumn(output_col, F.split(normalized_col, ','))


# Definir la función para capitalizar la primera letra, limpiar espacios y reemplazar '_'
def clean_text(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.trim(F.regexp_replace(F.col(input_col), '_', ' '))
    )

# funcion para extraer el contenido entre corchetes
def extract_string_content(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.when(
            F.col(input_col).rlike(r'\[.*?\]'),  # Si contiene corchetes
            F.regexp_replace(  # Remover las comillas dobles después de extraer el contenido
                F.regexp_extract(F.col(input_col), r'\[(.*?)\]', 1),
                r'"', ''  # Reemplazar todas las comillas dobles por un string vacío
            )
        ).otherwise(F.col(input_col))  # Si no tiene corchetes, dejar el valor original
    )



####### FIRST SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE

df = table
# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]
# completely_null_columns = [c for c in df.columns if non_null_counts[c] == 0]


# select the founded fields
table_cols_filt = df.select(non_completely_null_columns).\
  withColumnRenamed("created_dt",'created_date').\
  withColumn("created_date",F.to_date('created_date'))



##### OTHER COLUMNS TO REMOVE
columns_to_remove = ['dept_store_regional','etl_batch_id','mdm_source','sys_created_by','beauty_supply_store','category','product_category']

##### Filter columns that contain the string '_dt' and the ones in columns_to_remove
columns_wt_dt = [col for col in table_cols_filt.columns if '_dt' not in col and col not in columns_to_remove]

# count of not-enterely null columns
# print(len(columns_wt_dt)) ## 51
#print(non_completely_null_columns)


# select the list of columns to keep
table_cols_filt = table_cols_filt.select(columns_wt_dt)
table_cols_filt.createOrReplaceTempView("table_cols_filt_vw")

# call the expand array function twice
table_exploded = expand_arrays(table_cols_filt) # explode first array levels
table_exploded = expand_arrays(table_exploded) # explode second array levels

table_exploded.createOrReplaceTempView("table_exploded_vw")
# print(table_exploded.count()) # 13155657



# SECOND SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE

df = table_exploded

# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# select the list of columns to keep
table_pre_unpivot = df.select(non_completely_null_columns).\
  drop('analysis_exact_age','analysis_calculated_age')

table_pre_unpivot.createOrReplaceTempView("table_pre_unpivot_vw")



###  UNPIVOT DEF_table
# 1. Identificar dinámicamente las columnas de atributos
non_attributes = ["brand_code","brand_country",'created_date',id_key]
attributes = [c for c in table_pre_unpivot.columns if c not in non_attributes] 

# 2. Crear la expresión para el unpivot con stack()
num_atributos = len(attributes)
stack_expr = ", ".join(
    [f"'{col}', {col}" for col in attributes]
)

# 3. Realizar el unpivot usando `stack()`
table_unpivot = table_pre_unpivot.select(
    *non_attributes,
    F.expr(f"stack({num_atributos}, {stack_expr}) as (attribute, value)")).distinct()


table_unpivot.createOrReplaceTempView("table_unpivot_vw")



## SE AGRUPAN LOS IDS POR ATRIBUTO Y VALOR
table_unpivot_gp = table_unpivot.groupBy("attribute",'value').agg(
        F.countDistinct(id_key).alias("id_counts"),
        #F.min("created_date").alias("min_date"),
        F.max("created_date").alias("max_date")
    )

# print(table_unpivot_gp.count())
table_unpivot_gp.createOrReplaceTempView("table_unpivot_gp_vw")



## asigna nuevos nombres a los atributos de las tablas de acuerdo con la lkp (primer paso para poder cruzarla)
data = [
    ('analysis_analysis_clinical_sign_clinical_sign','CLINICAL_SIGN'),
    ('analysis_analysis_clinical_sign_sign_type','SIGN_TYPE'),
    ('analysis_analysis_clinical_sign_zone','ZONE'),
    ('analysis_analysis_concern_concern','CONCERN'),
    ('analysis_analysis_concern_concern_type','CONCERN_TYPE'), 
    ('analysis_analysis_concern_zone','ZONE'),
    ('analysis_analysis_type','ANALYSIS_TYPE'),
    ('channel_preference_product_purchase_channel','PRODUCT_PURCHASE_CHANNEL'),
    ('concern_improvement_goal_concern','CONCERN'),
    ('concern_zone','ZONE'),
    ('desired_color_finish','HAIR_COLOR_FINISH'),
    ('desired_color_permanence','HAIR_COLOR_PERMANENCE'), 
    ('desired_cover','HAIR_COLOR_COVER'), 
    ('fragrance_routine_perfume_moment','PERFUME_MOMENT'),
    ('fragrance_priority','FRAGRANCE_MOTIVATION'),
    ('hair_care_product_used','HAIRCARE_PRODUCT'),
    ('hair_routine_heating_heating_tool','HEATING_TOOL'),
    ('improvement_goal_concern','CONCERN'),
    ('last_hair_color_service','HAIR_COLOR_SERVICE'),
    ('last_hair_style_look','HAIR_STYLE_LOOK'),
    ('last_skincare_medical_treatment','LAST_SKINCARE_PROFESSIONAL_TREATMENT'),
    ('left_eye_color','EYE_COLOR'),
    ('look_occasion_makeup','MAKEUP_LOOK_OCCASION'),
    ('makeup_priority','MAKEUP_EXPECTATION'),
    ('natural_hair_color','HAIR_COLOR'),
    ('right_eye_color','EYE_COLOR'),
    ('skin_sensitivity_skin_sensitivity','SKIN_SENSITIVITY'),
    ('skin_sensitivity_zone','ZONE'),
    ('skin_type_skin_type','SKIN_TYPE'),
    ('desired_color_finish','ZONE'), 
    ('desired_style','HAIR_STYLE'), 
    ('skin_type_zone','ZONE')
]


# Crear el DataFrame
columns = ["attribute", "attribute_k"]
attributes_mapping = spark.createDataFrame(data, columns)
attributes_mapping.createOrReplaceTempView("attributes_mapping_vw")



# Importamos la lkp
bmdm_lkp = spark.table("prod_latam_catalog.crm_reporting.lkp_bmdm_reference")

# some transformations
bmdm_lkp = bmdm_lkp.withColumn('reference_name', F.lower('reference_name')).\
  select('reference_name','reference_value').\
  orderBy('reference_name','reference_value')	

bmdm_lkp.createOrReplaceTempView("bmdm_lkp_vw")

# display(bmdm_lkp)



## CRUCE CON LA LKP
table_result = spark.sql("""
with merge_1 as (
select
  a.*,
  CASE WHEN b.attribute_k IS NULL THEN a.attribute
    ELSE lower(b.attribute_k) END as attribute_k
from table_unpivot_gp_vw a
left join attributes_mapping_vw b
  on a.attribute = b.attribute
)

select 
  a.*,
  c.reference_value as value_lkp,
  case when c.reference_value is null then 'wrong' else 'ok' end as status
from merge_1 a
left join bmdm_lkp_vw c
  on a.attribute_k = c.reference_name
  and a.value = c.reference_value
""")

# Filtrar valores que tienen 6 o menos caracteres numericos para quitar hasheados
# table_result = table_result.filter(~F.col("value").rlike(r'(.*[0-9]){6,}'))

table_result.createOrReplaceTempView("table_result_vw")


In [0]:
# Aqui diligencia el atributo que deseas filtrar
atributo = 'analysis_analysis_concern_concern'

# valores incorrectos sin hasheados
tmp = table_result.filter((F.col('attribute') == atributo) &
                          (F.col('status') == 'wrong') &
                          (F.col('value').isNotNull())).\
        orderBy(F.desc("id_counts"))
display(tmp)


attribute,value,id_counts,max_date,attribute_k,value_lkp,status
analysis_analysis_concern_concern,null,6533612,2025-03-13,concern,null,wrong
analysis_analysis_concern_concern,~,11537,2024-06-08,concern,null,wrong
analysis_analysis_concern_concern,Hydration,4836,2024-04-22,concern,null,wrong
analysis_analysis_concern_concern,Color Fading,3899,2024-06-10,concern,null,wrong
analysis_analysis_concern_concern,Wriknles,3434,2024-04-22,concern,null,wrong
analysis_analysis_concern_concern,Imperfections,2155,2024-03-19,concern,null,wrong
analysis_analysis_concern_concern,Tighness,1240,2024-03-19,concern,null,wrong
analysis_analysis_concern_concern,Combination,240,2024-04-22,concern,null,wrong
analysis_analysis_concern_concern,Blonde,192,2024-06-07,concern,null,wrong
analysis_analysis_concern_concern,Fine Lines,174,2024-06-02,concern,null,wrong
